# ResNet基準：ERM・DROのモデル構成・学習手順

ResNet（M01 ERM・M02 DRO）、Spatial S34（M03 ERM・M04/M06 DRO）、HyperAdapt（M07 ERM・M08/M10 DRO）の8条件を比較。ERMは1-stage、DROは1-stage／凍結2-stage。global class weight、seed 42–44、race-known test、best validation AUROCに固定する。

2-stageは同seed M01親でbackbone・元classifier凍結。30対30+30epochの手順全体の比較で、DRO単独・凍結単独の効果とは呼ばない。M01対M02には学習GPU差もある。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import rootutils
from IPython.display import display

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import STYLE_SHEET, study_dir  # noqa: E402
from analysis.archive.race_known_architecture_objective_comparison.comparison import (  # noqa: E402
    GROUP_LABELS,
    METRICS,
    evaluate,
    group_ids,
    paired_deltas,
    plot_q,
    prediction_command,
    read_predictions,
    read_q_history,
    select_runs,
)

plt.style.use(STYLE_SHEET)
PACKAGE = study_dir("race_known_architecture_objective_comparison")
TOPIC = "06_resnet_baselines_vs_groupdro_architectures"
SEEDS = [42, 43, 44]
CONDITIONS = {
    "M01": "resnet-chexpert-race-known",
    "M02": "resnet-chexpert-age-race-group-dro-race-known",
    "M03": "spatial-lora-chexpert-race-known",
    "M04": "spatial-lora-chexpert-age-race-group-dro-race-known",
    "M06": "spatial-lora-chexpert-from-resnet-age-race-group-dro-race-known",
    "M07": "hyperadapt-chexpert-race-known",
    "M08": "hyperadapt-chexpert-age-race-group-dro-race-known",
    "M10": "hyperadapt-chexpert-from-resnet-age-race-group-dro-race-known",
}
LABELS = {
    "M01": "ResNet ERM",
    "M02": "ResNet DRO",
    "M03": "Spatial ERM",
    "M04": "Spatial DRO 1-stage",
    "M06": "Spatial DRO 2-stage",
    "M07": "HyperAdapt ERM",
    "M08": "HyperAdapt DRO 1-stage",
    "M10": "HyperAdapt DRO 2-stage",
}
PAIR_SETS = {
    "ResNet ERM baseline": [["M01", other] for other in ["M02", "M03", "M04", "M06", "M07", "M08", "M10"]],
    "ResNet DRO baseline": [["M02", other] for other in ["M04", "M06", "M08", "M10"]],
    "Within-model objective": [["M03", "M04"], ["M07", "M08"]],
    "Within-model workflow": [["M04", "M06"], ["M08", "M10"]],
}
PAIRS = [pair for pairs in PAIR_SETS.values() for pair in pairs]
CORE = ["auroc", "worst_auroc", "bacc", "worst_bacc"]
FAIRNESS = ["auroc_gap", "bacc_gap", "Eopp0", "Eopp1", "Eodds"]
PLOT_STYLES = [
    ("ERM 1-stage", {"M01": 0, "M03": 1, "M07": 2}, -0.16, "tab:green"),
    ("DRO 1-stage", {"M02": 0, "M04": 1, "M08": 2}, 0.0, "tab:blue"),
    ("DRO frozen 2-stage", {"M06": 1, "M10": 2}, 0.16, "tab:orange"),
]
RESULTS = PACKAGE / "results" / TOPIC
FIGURES = PACKAGE / "figures" / TOPIC
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

## 入力

成功run・checkpoint・train/valのSHA256を学習記録と照合し、test CSVの現在のSHA256を保存する。cache欠損時は提示コマンドをrepo rootで実行する（GPU自動起動なし）。

In [ ]:
provenance, frame, directories = select_runs(PACKAGE, CONDITIONS, SEEDS)
expected_objectives = {
    condition: "erm" if condition in {"M01", "M03", "M07"} else "group_dro_global" for condition in CONDITIONS
}
assert provenance.objective.eq(provenance.condition.map(expected_objectives)).all()
assert provenance.class_weight_scope.eq("global").all()
assert provenance.freeze_backbone.eq(provenance.condition.isin(["M06", "M10"])).all()
provenance.to_csv(RESULTS / "provenance.csv", index=False)
display(provenance[["condition", "seed", "run_id", "checkpoint", "objective", "freeze_backbone", "parent_run"]])
command = prediction_command(PACKAGE, directories, device="cuda:0")
print(command or "全予測cacheが有効")

## 全体性能・動作点

AUROCとBACC、worst-groupの両指標、感度TPR・特異度TNRを併記する。予測は固定argmax（確率の同率を除き0.5閾値）。平均±標準偏差は3seed間のばらつき。

In [ ]:
caches = read_predictions(PACKAGE, directories, frame)
summary, subgroup = evaluate(caches, frame)
summary.to_csv(RESULTS / "metrics_by_seed.csv", index=False)
subgroup.to_csv(RESULTS / "subgroup_by_seed.csv", index=False)
condition_stats = summary.groupby("condition", sort=False)[METRICS + ["tpr", "tnr"]].agg(["mean", "std"])
condition_stats.to_csv(RESULTS / "condition_mean_std.csv")
display(condition_stats[CORE + ["tpr", "tnr"]].rename(index=LABELS).round(4))
fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")
for ax, metric in zip(axes.flat, CORE, strict=True):
    baseline = condition_stats.loc["M01", metric]
    ax.axhline(baseline["mean"], color="grey", linestyle="--", label="ResNet ERM")
    ax.axhspan(baseline["mean"] - baseline["std"], baseline["mean"] + baseline["std"], color="grey", alpha=0.1)
    for label, positions, offset, color in PLOT_STYLES:
        stats = condition_stats.loc[list(positions), metric]
        ax.errorbar(
            [x + offset for x in positions.values()],
            stats["mean"],
            yerr=stats["std"],
            fmt="o",
            capsize=3,
            label=label,
            color=color,
        )
    ax.set_xticks([0, 1, 2], ["ResNet", "Spatial S34", "HyperAdapt"])
    ax.set_title(metric)
    ax.legend(fontsize=8)
fig.savefig(FIGURES / "overall_and_worst_performance.png", dpi=160)
plt.show()

## 同seedの差

比較条件−基準を、ERM基準／DRO基準／同モデルの目的関数差／手順差に分ける。平均±標準偏差を表示し、seed別の全指標はCSVに保存する。

In [ ]:
delta = paired_deltas(summary, PAIRS)
delta.to_csv(RESULTS / "paired_deltas.csv", index=False)
delta_stats = delta.groupby("pair", sort=False)[METRICS].agg(["mean", "std"])
delta_stats.to_csv(RESULTS / "paired_delta_mean_std.csv")
for scope, pairs in PAIR_SETS.items():
    print(scope)
    pair_names = [f"{left} → {right}" for left, right in pairs]
    display(delta_stats.loc[pair_names, CORE].round(4))
workflow_names = [f"{left} → {right}" for left, right in PAIR_SETS["Within-model workflow"]]
display(delta[delta.pair.isin(workflow_names)][["pair", "seed"] + CORE].round(4))

## 8群の性能・公平性

年齢<65／≥65 × race4区分。件数・陽性/陰性数を添え、未定義指標はNaN。gap/Eopp/Eoddsは小さい方がよいが、worst改善かbest低下かを分けて読む。EoddsはTPR gapとFPR gapの平均。

In [ ]:
counts = subgroup.query("condition == 'M01' and seed == 42")[["group_id", "group", "n", "n_positive", "n_negative"]]
counts.to_csv(RESULTS / "group_counts.csv", index=False)
print(f"test全体: {len(frame)}, 8群対象: {(group_ids(frame) >= 0).sum()}, 属性欠損除外: {(group_ids(frame) < 0).sum()}")
display(counts.set_index("group"))
group_stats = subgroup.groupby(["group", "condition"], sort=False)[["auroc", "bacc", "tpr", "tnr"]].agg(["mean", "std"])
group_stats.to_csv(RESULTS / "subgroup_mean_std.csv")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), layout="constrained")
for ax, metric in zip(axes, ["auroc", "bacc"], strict=True):
    print(f"8-group {metric}: 3-seed mean")
    table = subgroup.pivot_table(index="group", columns="condition", values=metric, aggfunc="mean")
    display(table.reindex(GROUP_LABELS)[list(CONDITIONS)].round(4))
    for condition, label in LABELS.items():
        stats = subgroup[subgroup.condition == condition].groupby("group_id")[metric].agg(["mean", "std"])
        ax.errorbar(stats.index, stats["mean"], yerr=stats["std"], marker="o", capsize=2, label=label)
    ax.set_xticks(range(8), GROUP_LABELS, rotation=65)
    ax.set_title(f"8-group {metric}")
axes[1].legend(fontsize=8)
fig.savefig(FIGURES / "subgroup_auroc_bacc.png", dpi=160)
plt.show()

## 公平性指標

AUROC gap・BACC gap・Eopp0（TNR gap）・Eopp1（TPR gap）は小さい方がよい。8群間で算出し、図は平均±標準偏差。Eoddsも表に併記。gap縮小はworst改善かbest低下かを併せて確認する。

In [ ]:
display(condition_stats[FAIRNESS].rename(index=LABELS).round(4))
condition_stats[FAIRNESS].to_csv(RESULTS / "fairness_mean_std.csv")
display(delta_stats[FAIRNESS].round(4))
delta_stats[FAIRNESS].to_csv(RESULTS / "fairness_delta_mean_std.csv")
fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")
for ax, metric in zip(axes.flat, FAIRNESS[:4], strict=True):
    baseline = condition_stats.loc["M01", metric]
    ax.axhline(baseline["mean"], color="grey", linestyle="--", label="ResNet ERM")
    ax.axhspan(baseline["mean"] - baseline["std"], baseline["mean"] + baseline["std"], color="grey", alpha=0.1)
    for label, positions, offset, color in PLOT_STYLES:
        stats = condition_stats.loc[list(positions), metric]
        ax.errorbar(
            [x + offset for x in positions.values()],
            stats["mean"],
            yerr=stats["std"],
            fmt="o",
            capsize=3,
            label=label,
            color=color,
        )
    ax.set_xticks([0, 1, 2], ["ResNet", "Spatial S34", "HyperAdapt"])
    ax.set_title(f"{metric} (lower is better)")
    ax.legend(fontsize=8)
fig.savefig(FIGURES / "fairness_gaps_eopp.png", dpi=160)
plt.show()

## qのepoch推移

epoch末qをseed別に表示。破線はbest checkpoint epoch（0始まり）。q_mean・weighted CE・countもCSVに保存。qはBACCではなくweighted CEに反応する。

In [ ]:
q_history = read_q_history(directories, provenance)
q_history.to_csv(RESULTS / "q_history.csv", index=False)
for panel, conditions in [
    ("resnet", ["M02"]),
    ("spatial", ["M04", "M06"]),
    ("hyperadapt", ["M08", "M10"]),
]:
    selected = q_history[q_history.condition.isin(conditions)]
    fig = plot_q(selected, LABELS)
    fig.savefig(FIGURES / f"q_end_{panel}.png", dpi=160)
    plt.show()
q_best = q_history[q_history.epoch == q_history.best_epoch].copy()
q_best.to_csv(RESULTS / "q_at_best.csv", index=False)

## TODO

- subgroup class weight・Spatial stage1–4は別比較。3seedのみで方式の優劣を断定しない。
- 群別weighted CEのtrain/val診断、validationでの閾値・checkpoint選択、患者単位の不確実性を確認。testで条件を選び直さない。